In [1]:
import pathlib
import re

import pandas as pd
import swifter

from functools import lru_cache
from rdkit import Chem

In [2]:
cwd = pathlib.Path().absolute()
quantum_green = pathlib.Path("/home/shared/projects/quantum_green")
paper_dir = quantum_green / "paper" / "figure"
data_dir = cwd.parent.parent / "data" / "thermo"

In [3]:
pd.set_option("display.max_columns", None)


def head(df, n=2):
    display(df.head(n))
    print(f"{len(df)} rows × {len(df.columns)} columns")


@lru_cache(maxsize=None)
def canonical_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    for atom in mol.GetAtoms():
        atom.SetAtomMapNum(0)
    mol = Chem.RemoveHs(mol)
    return Chem.MolToSmiles(mol, isomericSmiles=True)


def get_rxn_smi(row):
    return (
        row["r1_smiles"]
        + "."
        + row["r2_smiles"]
        + ">>"
        + row["p1_smiles"]
        + "."
        + row["p2_smiles"]
    )

Read processed thermo data

In [4]:
thermo_data = pd.read_csv(
    paper_dir
    / "section_3_2_1_thermo"
    / "quantum_green_species_data_24august14_dft_opted_dlpno_sp_thermos.csv"
)
head(thermo_data)

,species_dft_log_source_utf_8_sha512,asmi,multiplicity,std_xyz_str,species_dft_frequencies,species_dft_hartreefock_energy_hartree,species_dlpno_sp_hartree,p_thermo,m_thermo,p_H298,m_H298,p_S298,m_S298,p_Cp300,m_Cp300,p_Cp400,m_Cp400,p_Cp500,m_Cp500,p_Cp600,m_Cp600,p_Cp800,m_Cp800,p_Cp1000,m_Cp1000,p_Cp1500,m_Cp1500
0,42d67c486eb53d22486a5d1da768e644bf644469198894...,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[N:4]...,2,17\n\nC 1.687648 -1.481461 0.769379\nC 1.14635...,"[69.0965, 133.1139, 185.0048, 247.5052, 269.23...",-434.321489,-434.224944,"Wilhoit(Cp0=(33.2579,'J/(mol*K)'), CpInf=(407....","Wilhoit(Cp0=(33.2579,'J/(mol*K)'), CpInf=(407....",89171.888145,92902.845218,384.472728,384.472728,138.524841,138.524841,175.864528,175.864528,208.654500,208.654500,236.432193,236.432193,278.945081,278.945081,308.548112,308.548112,350.979140,350.979140
1,7667c6f680d9065479d515666e4c3246f65704e4926140...,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[O:4]...,2,21\n\nC -2.800669 0.461226 0.774726\nC -1.7902...,"[40.2091, 100.6031, 169.247, 232.8355, 250.021...",-403.429122,-403.298238,"Wilhoit(Cp0=(33.2579,'J/(mol*K)'), CpInf=(507....","Wilhoit(Cp0=(33.2579,'J/(mol*K)'), CpInf=(507....",102076.181436,104632.984293,397.355792,397.355792,153.423259,153.423259,200.307331,200.307331,241.584659,241.584659,276.984827,276.984827,332.331738,332.331738,371.824840,371.824840,429.622128,429.622128


339428 rows × 27 columns


In [5]:
thermo_data["p_thermo"].isna().sum().item()

2806

Since the thermo data has some missing values, we need to filter them out.

Also, we convert the smiles to canonical form.

Finally, we remove molecule `N=C=S` and its associated radical `[N]=C=S` from the dataset due to an known issue with the parsing of frequency data from the Gaussian log file.

In [6]:
filtered_thermo_data = thermo_data[thermo_data["p_thermo"].notna()].copy()
filtered_thermo_data["smiles"] = filtered_thermo_data["asmi"].swifter.apply(
    canonical_smiles
)
filtered_thermo_data.drop(
    filtered_thermo_data[
        filtered_thermo_data["smiles"].isin(["[N]=C=S", "N=C=S"])
    ].index,
    inplace=True,
)
filtered_thermo_data.reset_index(drop=True, inplace=True)
head(filtered_thermo_data)

Pandas Apply:   0%|          | 0/336622 [00:00<?, ?it/s]

,species_dft_log_source_utf_8_sha512,asmi,multiplicity,std_xyz_str,species_dft_frequencies,species_dft_hartreefock_energy_hartree,species_dlpno_sp_hartree,p_thermo,m_thermo,p_H298,m_H298,p_S298,m_S298,p_Cp300,m_Cp300,p_Cp400,m_Cp400,p_Cp500,m_Cp500,p_Cp600,m_Cp600,p_Cp800,m_Cp800,p_Cp1000,m_Cp1000,p_Cp1500,m_Cp1500,smiles
0,42d67c486eb53d22486a5d1da768e644bf644469198894...,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[N:4]...,2,17\n\nC 1.687648 -1.481461 0.769379\nC 1.14635...,"[69.0965, 133.1139, 185.0048, 247.5052, 269.23...",-434.321489,-434.224944,"Wilhoit(Cp0=(33.2579,'J/(mol*K)'), CpInf=(407....","Wilhoit(Cp0=(33.2579,'J/(mol*K)'), CpInf=(407....",89171.888145,92902.845218,384.472728,384.472728,138.524841,138.524841,175.864528,175.864528,208.654500,208.654500,236.432193,236.432193,278.945081,278.945081,308.548112,308.548112,350.979140,350.979140,CC1C[N]C(=N)N1C=O
1,7667c6f680d9065479d515666e4c3246f65704e4926140...,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[O:4]...,2,21\n\nC -2.800669 0.461226 0.774726\nC -1.7902...,"[40.2091, 100.6031, 169.247, 232.8355, 250.021...",-403.429122,-403.298238,"Wilhoit(Cp0=(33.2579,'J/(mol*K)'), CpInf=(507....","Wilhoit(Cp0=(33.2579,'J/(mol*K)'), CpInf=(507....",102076.181436,104632.984293,397.355792,397.355792,153.423259,153.423259,200.307331,200.307331,241.584659,241.584659,276.984827,276.984827,332.331738,332.331738,371.824840,371.824840,429.622128,429.622128,CC1COC(C2C[N]2)C1


336620 rows × 28 columns


Read the processed kinetics data and convert the species smiles to canonical form.

Make sure that all species in the kinetics data are present in the thermo data.

In [7]:
rate_data = pd.read_csv(
    paper_dir
    / "section_3_2_3_rate"
    / "quantum_green_ts_data_24september17_dft_opted_dlpno_sp_rates.csv"
)
for species in ["r1", "r2", "p1", "p2"]:
    rate_data[f"{species}_smiles"] = rate_data[f"{species}smi"].swifter.apply(
        canonical_smiles
    )
head(rate_data)

Pandas Apply:   0%|          | 0/166166 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/166166 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/166166 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/166166 [00:00<?, ?it/s]

,Unnamed: 0,ts_dft_log_source_utf_8_sha512,multiplicity,r1smi,r2smi,p1smi,p2smi,std_xyz_str,r1_matched_std_xyz_str,r2_matched_std_xyz_str,p1_matched_std_xyz_str,p2_matched_std_xyz_str,ts_dft_frequencies,r1_dft_frequencies,r2_dft_frequencies,p1_dft_frequencies,p2_dft_frequencies,ts_dft_hartreefock_energy_hartree,r1_dft_hartreefock_energy_hartree,r2_dft_hartreefock_energy_hartree,p1_dft_hartreefock_energy_hartree,p2_dft_hartreefock_energy_hartree,ts_dlpno_sp_hartree,r1_dlpno_sp_hartree,r2_dlpno_sp_hartree,p1_dlpno_sp_hartree,p2_dlpno_sp_hartree,neg_freq,kinetics_low,k_298,p_reaction_low,m_reaction_low,p_rev_kinetics_low,m_rev_kinetics_low,kinetics_high,p_reaction_high,m_reaction_high,p_rev_kinetics_high,m_rev_kinetics_high,low_A,p_rev_low_A,m_rev_low_A,low_Ea,p_rev_low_Ea,m_rev_low_Ea,high_A,p_rev_high_A,m_rev_high_A,high_Ea,p_rev_high_Ea,m_rev_high_Ea,p_deltaHrxn298,m_deltaHrxn298,p_deltaGrxn298,m_deltaGrxn298,r1_smiles,r2_smiles,p1_smiles,p2_smiles
0,0,4c648a435eb66f655a9afa7aebe1b6288cfbe5d4377ab7...,2,[O:1]([O:2])[H:3],[C:4]1([C:5]([C:8]([C:10](=[O:22])[H:23])([H:1...,[C:4]1([C:5]([C:8]([C:10](=[O:22])[H:23])([H:1...,[O:1]([O:2][H:11])[H:3],23\n\nO -0.296608 2.50034 -0.42834\nO 0.12862 ...,3\n\nO 0.055113 0.705073 0.0\nO 0.055113 -0.59...,20\n\nC 0.216362 1.782055 0.054566\nC -0.13150...,19\n\nC -0.008753 1.669215 -0.212931\nC 0.0510...,4\n\nO 0.0 0.708624 -0.054351\nO -0.0 -0.70862...,"[-1819.2074, 54.014, 69.9409, 78.7383, 95.0168...","[1268.6637, 1466.9774, 3673.0462]","[54.4468, 75.2434, 102.0769, 166.2486, 229.237...","[38.6623, 46.6427, 70.6254, 124.3627, 195.2907...","[355.51, 1029.1913, 1349.894, 1491.2134, 3851....",-499.513846,-150.736553,-348.800739,-348.140437,-151.376196,-499.420713,-150.773531,-348.675816,-348.013214,-151.4225,-1819.2074,"Arrhenius(A=(1.93009e+10,'cm^3/(mol*s)'), n=0,...",1.961906e-07,[O]O + CC(CC=O)C1CC1 <=> C[C](CC=O)C1CC1 + OO,[O]O + CC(CC=O)C1CC1 <=> C[C](CC=O)C1CC1 + OO,"Arrhenius(A=(23353.4,'m^3/(mol*s)'), n=-0.4985...","Arrhenius(A=(23353.4,'m^3/(mol*s)'), n=-0.4985...","Arrhenius(A=(2.46143e+12,'cm^3/(mol*s)'), n=0,...",[O]O + CC(CC=O)C1CC1 <=> C[C](CC=O)C1CC1 + OO,[O]O + CC(CC=O)C1CC1 <=> C[C](CC=O)C1CC1 + OO,"Arrhenius(A=(2.97825e+06,'m^3/(mol*s)'), n=-0....","Arrhenius(A=(2.97825e+06,'m^3/(mol*s)'), n=-0....",19300.901420,2.335344e+04,2.335344e+04,64517.736480,30906.955270,30906.955270,2.461429e+06,2.978245e+06,2.978245e+06,98788.712643,65177.931433,65177.931433,34171.85163,34171.85163,26963.412254,26963.412254,[O]O,CC(CC=O)C1CC1,C[C](CC=O)C1CC1,OO
1,1,3fcdf61497a20d2e765655162f9a50bd8067a6ae90e90e...,2,[O:1]([O:2])[H:3],[C:4]1([O:11][C:5]([C:8]([O:12][C:10]([H:24])(...,[C:4]1([O:11][C:5]([C:8]([O:12][C:10]([H:25])[...,[O:1]([O:2][H:24])[H:3],26\n\nO 2.669129 -1.916241 0.102074\nO 3.62579...,3\n\nO 0.055113 0.705073 0.0\nO 0.055113 -0.59...,23\n\nC 2.355263 -1.592675 0.183484\nO 2.52464...,22\n\nC 1.79229 -1.9024 0.57597\nO 1.564071 -1...,4\n\nO 0.0 0.708624 -0.054351\nO -0.0 -0.70862...,"[-1780.9655, 22.7631, 39.1821, 48.2943, 53.310...","[1268.6637, 1466.9774, 3673.0462]","[41.7173, 46.7835, 79.0284, 116.919, 165.6573,...","[41.6898, 69.9592, 110.6483, 138.1279, 181.957...","[355.51, 1029.1913, 1349.894, 1491.2134, 3851....",-575.835651,-150.736553,-425.124273,-424.462568,-151.376196,-575.755980,-150.773531,-425.012518,-424.350164,-151.4225,-1780.9655,"Arrhenius(A=(4.46942e+10,'cm^3/(mol*s)'), n=0,...",1.102303e-07,[O]O + COCC(C)OC1CC1 <=> [CH2]OCC(C)OC1CC1 + OO,[O]O + COCC(C)OC1CC1 <=> [CH2]OCC(C)OC1CC1 + OO,"Arrhenius(A=(2.07496e+06,'m^3/(mol*s)'), n=-0....","Arrhenius(A=(2.07496e+06,'m^3/(mol*s)'), n=-0....","Arrhenius(A=(6.07844e+12,'cm^3/(mol*s)'), n=0,...",[O]O + COCC(C)OC1CC1 <=> [CH2]OCC(C)OC1CC1 + OO,[O]O + COCC(C)OC1CC1 <=> [CH2]OCC(C)OC1CC1 + OO,"Arrhenius(A=(2.82196e+08,'m^3/(mol*s)'), n=-0....","Arrhenius(A=(2.82196e+08,'m^3/(mol*s)'), n=-0....",44694.157891,2.074960e+06,2.074960e+06,68134.787555,35997.063187,35997.063187,6.078441e+06,2.821962e+08,2.821962e+08,102752.6

166166 rows × 59 columns


In [8]:
all_smiles_in_rate_data = set().union(
    *[rate_data[f"{species}_smiles"] for species in ["r1", "r2", "p1", "p2"]]
)
len(all_smiles_in_rate_data)

274686

In [9]:
all_smiles_in_rate_data.issubset(filtered_thermo_data["smiles"])

True

Parse the Wilhoit model parameters from the thermo data.

Also, show that the `Cp0` parameter is the same for all species, so we can remove it from the dataset.

In [10]:
REGEX = re.compile(r'(\w+)\s*=\s*\(?\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)')


def parse_wilhoit_string(wilhoit_str):
    return {
        param_name: float(value)
        for param_name, value in re.findall(REGEX, wilhoit_str)
    }

In [11]:
wilhoit_df = pd.DataFrame(
    [
        parse_wilhoit_string(wilhoit_str)
        for wilhoit_str in filtered_thermo_data["p_thermo"]
    ]
)
head(wilhoit_df)

,Cp0,CpInf,a0,a1,a2,a3,H0,S0,B
0,33.2579,407.409,-1.60270,7.86347,-12.21810,3.797620e+00,-37.3533,-2025.56,300.00
1,33.2579,507.183,-7.96383,9.72081,-3.84389,7.901530e-07,-252.5980,-3207.62,1138.67


336620 rows × 9 columns


In [12]:
Cp0_counts = wilhoit_df["Cp0"].value_counts()
Cp0_counts

Cp0
33.2579    336620
Name: count, dtype: int64

In [13]:
wilhoit_df.drop(columns=["Cp0"], inplace=True)

Read dataset containing Zero Point Energy values for all species.

In [14]:
zpe_data = pd.read_pickle(
    quantum_green / "reactants" / "quantum_green_species_data_24march12b.pkl"
)
head(zpe_data)

,asmi,species_id,species_dft_log_path,species_dft_route_section,charge,multiplicity,species_dft_opt_max_steps,species_dft_opt_normal_termination_check,species_dft_opt_cpu_time,species_dft_opt_wall_time,species_dft_sum_of_electronic_and_thermal_enthalpies_hartree,species_dft_hartreefock_energy_hartree,species_dft_zpe_hartree,species_dft_e0_zpe_hartree,species_dft_gibbs_hartree,species_dft_scf_hartree,species_dft_frequency_modes,species_dft_std_forces,species_dft_opted_std_xyz,species_dft_opted_xyz_input_orientation,is_ts,batch_label,species_dft_log_filename_id,nasmi,matched_rxn_fingerprint,species_dft_log_source_utf_8_sha512,passed_connectivity_check,rdkit_canonical_nasmi,partent_rdkit_canonical_nasmi_radical_only,reaction_center_atom_map_num_radical_only,std_xyz_str,xyz_str,matched_parent_mol_sha512_radical_only,matched_parent_mol_asmi_radical_only,matched_parent_mol_reaction_center_atom_map_num_radical_only,species_dft_first_freq,species_dlpno_log_filename_id,species_dlpno_log_path,species_dlpno_sp_hartree,dlpno_batch_label,species_dft_frequencies
0,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[N:4]...,1,/home/gridsan/groups/RMG/Projects/Hao-Wei-Osca...,"P opt=(calcfc,maxcycle=128,noeig,nomicro,carte...",0,2,102,True,4100.1,1076.5,-434.175602,-434.321489,0.136748,-434.184741,-434.218451,434.321490,"[[[1.0, 6.0, -0.01, 0.18, 0.17], [2.0, 6.0, -0...","[[1.0, 6.0, 1.274e-05, 2.05e-06, 3.763e-06], [...","[[1.0, 6.0, 0.0, 1.687648, -1.481461, 0.769379...","[[1.0, 6.0, 0.0, 1.711519, -1.472917, 0.740063...",False,aug11b,id52810,CC1C[N]C(=N)N1C=O,"{('46e86ef9080f8c5b21f5c329d23f0d6c', 'p2_smi')}",42d67c486eb53d22486a5d1da768e644bf644469198894...,True,CC1C[N]C(=N)N1C=O,CC1CNC(=N)N1C=O,4.0,17\n\nC 1.687648 -1.481461 0.769379\nC 1.14635...,17\n\nC 1.711519 -1.472917 0.740063\nC 1.17193...,4f388644fa3d52bec962545fb1a65c4446aa8758d8834c...,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[N:4]...,16,69.0965,id52810,/home/gridsan/jburns/RMG_shared/Projects/Hao-W...,-434.224944,reactant_aug11b,"[69.0965, 133.1139, 185.0048, 247.5052, 269.23..."
1,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[O:4]...,3,/home/gridsan/groups/RMG/Projects/Hao-Wei-Osca...,"P opt=(calcfc,maxcycle=128,noeig,nomicro,carte...",0,2,126,True,12128.7,3218.9,-403.235873,-403.429122,0.183595,-403.245527,-403.280167,403.429122,"[[[1.0, 6.0, 0.21, 0.06, 0.18], [2.0, 6.0, 0.0...","[[1.0, 6.0, 5.513e-06, 2.7033e-05, 2.003e-06],...","[[1.0, 6.0, 0.0, -2.800669, 0.461226, 0.774726...","[[1.0, 6.0, 0.0, 2.701051, 1.034704, 0.04014],...",False,aug11b,id52729,CC1COC(C2C[N]2)C1,"{('5ddcee1ad82096fe7dcfd17162f30b09', 'p2_smi')}",7667c6f680d9065479d515666e4c3246f65704e4926140...,True,CC1COC(C2C[N]2)C1,CC1COC(C2CN2)C1,8.0,21\n\nC -2.800669 0.461226 0.774726\nC -1.7902...,21\n\nC 2.701051 1.034704 0.04014\nC 1.73068 -...,f0d1bccdf82215b9726f7ea9b5b912db1d1b0a4b8fd7e0...,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[O:4]...,20,40.2091,id52729,/home/gridsan/jburns/RMG_shared/Projects/Hao-W...,-403.298238,reactant_aug11b,"[40.2091, 100.6031, 169.247, 232.8355, 250.021..."


348258 rows × 41 columns


Collect the relevant data for each species.

In [15]:
thermo_df = pd.DataFrame(
    {
        "smiles": filtered_thermo_data["smiles"],
        "H298": filtered_thermo_data["p_H298"],
        "S298": filtered_thermo_data["p_S298"],
        "Cp300": filtered_thermo_data["p_Cp300"],
        "dlpno_sp_hartree": filtered_thermo_data["species_dlpno_sp_hartree"],
        "dft_zpe_scaled_hartree": 0.9723865877712031
        * filtered_thermo_data["smiles"].map(
            zpe_data.set_index("rdkit_canonical_nasmi")["species_dft_zpe_hartree"]
        ),
        **wilhoit_df.to_dict(orient="list"),
        "closed_shell": filtered_thermo_data["multiplicity"] == 1,
    }
)
head(thermo_df)

,smiles,H298,S298,Cp300,dlpno_sp_hartree,dft_zpe_scaled_hartree,CpInf,a0,a1,a2,a3,H0,S0,B,closed_shell
0,CC1C[N]C(=N)N1C=O,89171.888145,384.472728,138.524841,-434.224944,0.132972,407.409,-1.60270,7.86347,-12.21810,3.797620e+00,-37.3533,-2025.56,300.00,False
1,CC1COC(C2C[N]2)C1,102076.181436,397.355792,153.423259,-403.298238,0.178526,507.183,-7.96383,9.72081,-3.84389,7.901530e-07,-252.5980,-3207.62,1138.67,False


336620 rows × 15 columns


Save the closed-shell and open-shell species in separate files.

In [16]:
thermo_df[thermo_df["closed_shell"]].drop(columns=["closed_shell"]).sort_values(
    by="smiles"
).to_csv(
    data_dir / "quantumpioneer_thermo_dataset_closed_shell_species.csv", index=False
)

thermo_df[~thermo_df["closed_shell"]].drop(columns=["closed_shell"]).sort_values(
    by="smiles"
).to_csv(
    data_dir / "quantumpioneer_thermo_dataset_open_shell_species.csv", index=False
)

### Sanity checks

Confirm that the dataset has no missing values.

In [17]:
for col in thermo_df.columns:
    num_na = thermo_df[col].isna().sum()
    if num_na > 0:
        print(f"Column {col} has {num_na} missing values")

In [18]:
reaction_data = pd.read_pickle(
    paper_dir
    / "section_3_1_2_ts_char"
    / "quantum_green_ts_data_24july2c.pkl"
)
head(reaction_data)

,ts_dft_log_source_utf_8_sha512,dft_table_id,ts_dft_log_path,ts_dft_route_section,charge,multiplicity,ts_dft_opt_max_steps,ts_dft_opt_normal_termination_check,ts_dft_opt_cpu_time,ts_dft_opt_wall_time,ts_dft_sum_of_electronic_and_thermal_enthalpies_hartree,ts_dft_hartreefock_energy_hartree,ts_dft_zpe_hartree,ts_dft_e0_zpe_hartree,ts_dft_gibbs_hartree,ts_dft_scf_hartree,ts_dft_frequency_modes,ts_dft_std_forces,ts_dft_opted_std_xyz,ts_dft_opted_xyz_input_orientation,is_ts,batch_label,ts_dft_log_filename_id,rsmi,psmi,r1smi,r2smi,p1smi,p2smi,r1nasmi,r2nasmi,p1nasmi,p2nasmi,std_xyz_str,xyz_str,p1_reaction_center_atom_map_num,r1_reaction_center_atom_map_num,h_atom_map_num,r1_matched_sha512,r2_matched_sha512,p1_matched_sha512,p2_matched_sha512,r1_matched_smi,r1_matched_std_xyz_str,r2_matched_smi,r2_matched_std_xyz_str,p1_matched_smi,p1_matched_std_xyz_str,p2_matched_smi,p2_matched_std_xyz_str,connectivity_check_r1,connectivity_check_r2,connectivity_check_r1_dist,connectivity_check_r2_dist,r2_h_idx,r2_reacting_site_idx,r2_reacting_site_element,r2_reacting_site_h_bond_distance,r2_ts_reacting_site_h_bond_distance,r2_ts_reacting_site_h_bond_distance_pct_change,connectivity_check_p1,connectivity_check_p2,connectivity_check_p1_dist,connectivity_check_p2_dist,p2_h_idx,p2_reacting_site_idx,p2_reacting_site_element,p2_reacting_site_h_bond_distance,p2_ts_reacting_site_h_bond_distance,p2_ts_reacting_site_h_bond_distance_pct_change,rxn_smi,fingerprint,rxn_smi_original_order,rdmc_large_factor,rdmc_small_factor,neg_freq,neg_freq_z_score,rp_connectivity_check_true_sum,connectivity_check_r1_score,connectivity_check_r2_score,connectivity_check_p2_score,connectivity_check_p1_score,ho2_ts_h_mode,ho2_non_react_h_mode,ho2_ts_h_score,ho2_non_react_h_score,ho2_non_div_ts_mode_ratio,r1_dft_e0_zpe_hartree,r2_dft_e0_zpe_hartree,p1_dft_e0_zpe_hartree,p2_dft_e0_zpe_hartree,rcomplex_dft_e0_zpe_hartree,pcomplex_dft_e0_zpe_hartree,dft_fwd_barrier_e0_zpe_hartree,dft_rev_barrier_e0_zpe_hartree,dft_fwd_Hrxn_e0_zpe_hartree,dft_fwd_barrier_e0_zpe_kcal,dft_rev_barrier_e0_zpe_kcal,dft_fwd_Hrxn_e0_zpe_kcal,ts_dlpno_log_filename_id,ts_dlpno_xyz_tuple,ts_dlpno_log_path,ts_dlpno_sp_hartree,dft_opted_xyz_tuple_used_for_dlpno_calc,r1_dlpno_sp_hartree,r2_dlpno_sp_hartree,p1_dlpno_sp_hartree,p2_dlpno_sp_hartree,all_species_has_dlpno_sp,fwd_Hrxn_dlpno_sp_hartree,ts_and_species_has_dlpno_sp,fwd_barrier_dlpno_sp_hartree,rev_barrier_dlpno_sp_hartree,fwd_Hrxn_dlpno_sp_kcal,fwd_barrier_dlpno_sp_kcal,rev_barrier_dlpno_sp_kcal,r1_dft_log_path,r2_dft_log_path,p1_dft_log_path,p2_dft_log_path,r1_dlpno_log_path,r2_dlpno_log_path,p1_dlpno_log_path,p2_dlpno_log_path,r1_dft_hartreefock_energy_hartree,r2_dft_hartreefock_energy_hartree,p1_dft_hartreefock_energy_hartree,p2_dft_hartreefock_energy_hartree,r1_dft_zpe_hartree,r2_dft_zpe_hartree,p1_dft_zpe_hartree,p2_dft_zpe_hartree,ts_and_species_has_dft_zpe,species_has_dft_zpe,fwd_Hrxn_dft_zpe_hartree,fwd_barrier_dft_zpe_hartree,rev_barrier_dft_zpe_hartree,fwd_Hrxn_dlpno_sp_dft_zpe_hartree,fwd_barrier_dlpno_sp_dft_zpe_hartree,rev_barrier_dlpno_sp_dft_zpe_hartree,fwd_Hrxn_dlpno_sp_dft_zpe_kcal,fwd_barrier_dlpno_sp_dft_zpe_kcal,rev_barrier_dlpno_sp_dft_zpe_kcal,r1_dft_zpe_scaled_hartree,r2_dft_zpe_scaled_hartree,p1_dft_zpe_scaled_hartree,p2_dft_zpe_scaled_hartree,ts_dft_zpe_scaled_hartree,fwd_barrier_dlpno_sp_dft_zpe_scaled_hartree,rev_barrier_dlpno_sp_dft_zpe_scaled_hartree,fwd_Hrxn_dlpno_sp_dft_zpe_scaled_hartree,rev_Hrxn_dlpno_sp_dft_zpe_scaled_hartree,fwd_barrier_dlpno_sp_dft_zpe_scaled_kcal,rev_barrier_dlpno_sp_dft_zpe_scaled_kcal,fwd_Hrxn_dlpno_sp_dft_zpe_scaled_kcal,rev_Hrxn_dlpno_sp_dft_zpe_scaled_kcal,ts_dft_frequencies,r1_dft_frequencies,r2_dft_frequencies,p1_dft_frequencies,p2_dft_frequencies,smiles
0,4c648a435eb66f655a9afa7aebe1b6288cfbe5d4377ab7...,545983,/home/gridsan/groups/RMG/Projects/Habs/data/ts...,"P opt=(ts,calcall,maxcycle=64,noeig,nomicro,ca...",0,2,64,True,53637.5,1178.8,-499.312919,-499.513846,0.187656,-499.326190,-499.365824,4

167237 rows × 162 columns


In [19]:
species_data_from_reactions = pd.concat(
    [
        pd.DataFrame(
            {
                "smiles": reaction_data[f"{species}smi"].apply(canonical_smiles),
                "dlpno_sp_hartree": reaction_data[f"{species}_dlpno_sp_hartree"],
                "dft_zpe_scaled_hartree": reaction_data[
                    f"{species}_dft_zpe_scaled_hartree"
                ],
            }
        ).drop_duplicates(subset=["smiles"], keep="first")
        for species in ["r1", "r2", "p1", "p2"]
    ]
).drop_duplicates(subset=["smiles"], keep="first")
head(species_data_from_reactions)

,smiles,dlpno_sp_hartree,dft_zpe_scaled_hartree
0,[O]O,-150.773531,0.014197
143328,[O]OCO,-265.183227,0.048332


276170 rows × 3 columns


In [20]:
dlpno_sp_from_reactions = species_data_from_reactions[["smiles", "dlpno_sp_hartree"]]
dlpno_sp_from_reactions["species_dlpno_sp_hartree"] = dlpno_sp_from_reactions[
    "smiles"
].map(thermo_df.set_index("smiles")["dlpno_sp_hartree"])
(
    dlpno_sp_from_reactions["species_dlpno_sp_hartree"]
    - dlpno_sp_from_reactions["dlpno_sp_hartree"]
).abs().max()

0.0

In [21]:
zpe_from_reactions = species_data_from_reactions[["smiles", "dft_zpe_scaled_hartree"]]
zpe_from_reactions["species_dft_zpe_hartree"] = zpe_from_reactions["smiles"].map(
    zpe_data.set_index("rdkit_canonical_nasmi")["species_dft_zpe_hartree"]
)
zpe_from_reactions["ratio"] = (
    zpe_from_reactions["dft_zpe_scaled_hartree"]
    / zpe_from_reactions["species_dft_zpe_hartree"]
)
zpe_from_reactions["ratio"].min(), zpe_from_reactions["ratio"].max()

(0.972386587771203, 0.9723865877712032)

In [22]:
def cp_wilhoit(T, Cp0, CpInf, a0, a1, a2, a3, B):
    y = T / (T + B)
    poly_sum = a0 + a1 * y + a2 * y**2 + a3 * y**3
    Cp = Cp0 + (CpInf - Cp0) * y**2 * (1 + (y - 1) * poly_sum)
    return Cp


def calculate_cp_wilhoit(row):
    return cp_wilhoit(
        300,
        33.2579,
        row["CpInf"],
        row["a0"],
        row["a1"],
        row["a2"],
        row["a3"],
        row["B"],
    )


In [ ]:
Cp300_wilhoit = thermo_df.apply(calculate_cp_wilhoit, axis=1)

In [23]:
relative_diff = ((Cp300_wilhoit - thermo_df["Cp300"]) / thermo_df["Cp300"]).abs()
relative_diff.describe()

count    3.366200e+05
mean     2.575427e-06
std      2.570809e-06
min      3.001578e-11
25%      7.554501e-07
50%      1.717685e-06
75%      3.510134e-06
max      1.782951e-05
dtype: float64

In [25]:
assert relative_diff.max() < 1e-4